In [3]:

from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "NOTEBOOKS":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "DATA"
RAW_DIR = DATA_DIR / "RAW"
PROCESSED_DIR = DATA_DIR / "PROCESSED"
MODELS_DIR = PROJECT_ROOT / "MODELS"
OUTPUTS_DIR = PROJECT_ROOT / "OUTPUTS"
PLOTS_DIR = OUTPUTS_DIR / "PLOTS"

for d in [PROCESSED_DIR, MODELS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version)

from sklearn.model_selection import train_test_split

train_path = PROCESSED_DIR / "train_processed.csv"
test_path = PROCESSED_DIR / "test_processed.csv"

if not train_path.exists():
    raise FileNotFoundError("Run 03_preprocessing.ipynb first.")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

target = "machine_failure"
X_train = train_df.drop(columns=[target]).copy()
y_train = train_df[target].copy()
X_test = test_df.drop(columns=[target]).copy()
y_test = test_df[target].copy()

print("Input features:", X_train.columns.tolist())

Project root: C:\Users\sitar\Downloads\ASSIGNMENT_CODE\PREDICTIVE-MAINTENANCE
Python: 3.10.19 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 16:41:31) [MSC v.1929 64 bit (AMD64)]
Input features: ['type', 'air_temperature_[k]', 'process_temperature_[k]', 'rotational_speed_[rpm]', 'torque_[nm]', 'tool_wear_[min]', 'twf', 'hdf', 'pwf', 'osf', 'rnf']


In [4]:
def add_engineered_features(X):
    X = X.copy()
    # AI4I sensor relationships
    if {"rotational_speed_rpm", "torque_nm"}.issubset(X.columns):
        X["mechanical_power_proxy"] = X["rotational_speed_rpm"] * X["torque_nm"]
    if {"process_temperature_k", "air_temperature_k"}.issubset(X.columns):
        X["temperature_difference"] = X["process_temperature_k"] - X["air_temperature_k"]
    if {"tool_wear_min", "rotational_speed_rpm"}.issubset(X.columns):
        X["wear_speed_interaction"] = X["tool_wear_min"] * X["rotational_speed_rpm"]
    if {"torque_nm", "rotational_speed_rpm"}.issubset(X.columns):
        X["torque_speed_ratio"] = X["torque_nm"] / X["rotational_speed_rpm"].replace(0, np.nan)
        X["torque_speed_ratio"] = X["torque_speed_ratio"].replace([np.inf, -np.inf], np.nan)
    return X

X_train_fe = add_engineered_features(X_train)
X_test_fe = add_engineered_features(X_test)

# Fill any division-generated missing values using training medians
num_fe = X_train_fe.select_dtypes(include=np.number).columns
X_train_fe[num_fe] = X_train_fe[num_fe].fillna(X_train_fe[num_fe].median())
X_test_fe[num_fe] = X_test_fe[num_fe].fillna(X_train_fe[num_fe].median())

print("Features after engineering:", len(X_train_fe.columns))
print(X_train_fe.columns.tolist())

X_train_fe.to_csv(PROCESSED_DIR / "X_train_engineered.csv", index=False)
X_test_fe.to_csv(PROCESSED_DIR / "X_test_engineered.csv", index=False)
y_train.to_csv(PROCESSED_DIR / "y_train.csv", index=False)
y_test.to_csv(PROCESSED_DIR / "y_test.csv", index=False)

Features after engineering: 11
['type', 'air_temperature_[k]', 'process_temperature_[k]', 'rotational_speed_[rpm]', 'torque_[nm]', 'tool_wear_[min]', 'twf', 'hdf', 'pwf', 'osf', 'rnf']
